# GWB + IRN + CW in ATLAS

In [ ]:
import sys, json, glob
sys.path.append('../')

from enterprise.pulsar import FeatherPulsar

import numpy as np
import jax.numpy as jnp
import jax.random as jrandom
import matplotlib.pyplot as plt
import corner

from numpyro.infer import MCMC, NUTS

from ATLAS.data import PTA_Data
from ATLAS.nMatrix.base import WhiteCov
from ATLAS.model import model_maker
from ATLAS.psd_functions import powerlaw, hd_orf
from ATLAS.signals.factorized.base import Red, SuperSignal
from ATLAS.signals.correlated.base import Correlated
from ATLAS.signals.deterministic.base import Deterministic
from ATLAS.signals.deterministic.det_signals import cw_delay_evolve_float64

%load_ext autoreload
%autoreload 2

## Load Data

Load a subset of pulsars from NG15 and the corresponding white noise dictionary.

In [ ]:
NPSRS = 5

feather_files = sorted(glob.glob('../data/NG15/feathers/*.feather'))
psrs = [FeatherPulsar.read_feather(f) for f in feather_files[:NPSRS]]

with open('../data/NG15/15yr_wn_dict.json', 'r') as fin:
    noise_dict = json.load(fin)

## Settings

Specify the number of frequency bins and parameter bounds per signal.

In [ ]:
NFREQS_IRN = 30
NFREQS_GWB = 14
NFREQS_DET = 60

# power-law bounds [log10_A, gamma]
psd_lower, psd_upper = jnp.array([-20., 0.]), jnp.array([-4., 7.])

# CW source parameters: log10_mc, log10_fgw, cos_inc, psi, log10_h, cos_gwtheta, gwphi, phase0
cw_param_labels = [r"$\log_{10}\mathcal{M}_c$", r"$\log_{10}f_\mathrm{gw}$", r"$\cos\iota$", r"$\psi$",
                   r"$\log_{10}h$", r"$\cos\theta_\mathrm{gw}$", r"$\phi_\mathrm{gw}$", r"$\Phi_0$"]
cw_param_mins = np.array([7.2, -8.7, -1., 0, -18., -1., 0., 0.])
cw_param_maxs = np.array([9., -8.2, 1., np.pi, -12., 1., 2. * np.pi, 2. * np.pi])
cw_parameter_bounds = np.array([cw_param_mins, cw_param_maxs]).T

## Model

The linear timing model is analytically marginalised and white noise fixed to the NG15 noise dictionary.

In [ ]:
data = PTA_Data(psrs,
                num_gwb_bins = NFREQS_GWB,
                num_irn_bins = NFREQS_IRN,
                num_det_bins = NFREQS_DET,
                num_dm_bins  = None,
                fixed_white_noise_params = None,
                linear_timing  = True,
                marg_timing    = True,
                diag_white_cov = False,
                fixed_res      = False,
                noise_dict     = noise_dict)

wn = WhiteCov(data = data, stabilize_TNT = True)
wn_vec = wn.params_dict_to_vector(noise_dict)

In [ ]:
# define constituent signal / noise models
sig_unc = Red(name = 'unc', data = data, psd_function = powerlaw, nfreqs = NFREQS_IRN,
              lower_bound_psd = psd_lower, upper_bound_psd = psd_upper)
sig_cor = Correlated(name = 'cor', data = data, psd_function = powerlaw, orf_function = hd_orf,
                     nfreqs = NFREQS_GWB, lower_bound_psd = psd_lower, upper_bound_psd = psd_upper)
sig_cw  = Deterministic(name = 'det', data = data, get_delays_func = cw_delay_evolve_float64,
                        det_parameter_bounds = cw_parameter_bounds, nfreqs_det = NFREQS_DET)

# combine components into joint model
super_sig = SuperSignal(signal_list = [sig_unc, sig_cor, sig_cw],
                        signal_combination_string = "ltm|unc+cor->unc ; det",
                        data = data)

# get helper objects for faster posterior evaluation ("TNT", "TNr", etc.)
helpers = super_sig.get_helpers(reff = jnp.concat(data.raw_residuals), white_noise_params = wn_vec)
red_param_names = super_sig.model.get_param_names()

## Sample

`model_maker` is the NumPyro model with arguments passed through `mcmc.run`.

In [ ]:
nuts_kernel = NUTS(model = model_maker)
mcmc = MCMC(sampler = nuts_kernel,
            num_warmup = 500,
            num_samples = 1000)
mcmc.run(jrandom.key(200129),
         raw_residuals = None, super_sig = super_sig, marg_over_non_gwb = False, helpers = helpers)
samples = mcmc.get_samples()

In [ ]:
fig = corner.corner(data = np.array(samples['det_params']),
                    labels = cw_param_labels)

In [ ]:
gwb_idx = [red_param_names.index('gwb_log10_A'), red_param_names.index('gwb_gamma')]
gwb = np.asarray(samples['red_noise'])[:, gwb_idx]
fig, ax = plt.subplots(1, 2, figsize = (9, 3.5))
ax[0].hist(gwb[:, 0], bins = 30, density = True, histtype = 'step', lw = 2)
ax[0].set_xlabel(r'HD $\log_{10}A$')
ax[1].hist(gwb[:, 1], bins = 30, density = True, histtype = 'step', lw = 2)
ax[1].set_xlabel(r'HD $\gamma$')
plt.show()